# 03. RFM Segmentation

Сегментация пользователей по Recency, Frequency, Monetary и оценка LTV.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

orders = pd.read_csv(Path('../data/fact_orders.csv'), parse_dates=['order_ts'])
orders = orders[orders['order_status'] == 'delivered'].copy()

In [2]:
anchor = orders['order_ts'].max().normalize()
rfm = orders.groupby('user_id').agg(
    recency_days=('order_ts', lambda s: (anchor - s.max().normalize()).days),
    frequency_orders=('order_id', 'count'),
    monetary_rub=('basket_value_rub', 'sum')
).reset_index()

rfm['r_score'] = pd.qcut(rfm['recency_days'].rank(method='first'), 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency_orders'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary_rub'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

In [3]:
def segment(row):
    if row.r_score >= 4 and row.f_score >= 4 and row.m_score >= 4:
        return 'Core Users'
    if row.r_score >= 4 and row.f_score >= 3:
        return 'Active Loyal'
    if row.r_score <= 2 and row.f_score >= 4:
        return 'High Value at Risk'
    if row.r_score <= 2 and row.f_score <= 2:
        return 'Dormant'
    return 'Growth Potential'

rfm['segment'] = rfm.apply(segment, axis=1)
rfm.groupby('segment').agg(users=('user_id', 'count'), avg_monetary=('monetary_rub', 'mean')).sort_values('avg_monetary', ascending=False)


,users,avg_monetary
segment,,
Core Users,1534,2602.890091
High Value at Risk,1483,2250.842158
Active Loyal,1052,1353.972586
Growth Potential,4387,1181.746289
Dormant,1737,710.095993


In [4]:
fig = px.scatter(
    rfm,
    x='recency_days',
    y='frequency_orders',
    size='monetary_rub',
    color='segment',
    hover_data=['user_id', 'rfm_score'],
    title='RFM User Segments',
    labels={
        'recency_days': 'Days since last order',
        'frequency_orders': 'Number of orders',
        'monetary_rub': 'Order value, RUB',
        'segment': 'Segment',
    },
    category_orders={
        'segment': ['Core Users', 'Active Loyal', 'Growth Potential', 'High Value at Risk', 'Dormant'],
    },
)
fig.update_layout(template='plotly_white', legend_title_text='Segment')
fig.show()
